In [4]:
from ui.mainwindow import * # import ui .py file, adjust ui.foo for filename

# main Qt core imports
from PySide6.QtCore import QCoreApplication, Qt, QIODevice, QTimer
from PySide6.QtWidgets import QApplication, QMainWindow, QFileDialog, QPushButton, QComboBox
from PySide6.QtGui import QIcon, QAction, QCloseEvent

import ctypes # Windows exclusive, allows for unique icon assignment
myappid = 'int.maldi' # arbitrary string
ctypes.windll.shell32.SetCurrentProcessExplicitAppUserModelID(myappid)

# utils
from pathlib import Path
from time import *
import sys
from utils import FluigentController

# Windows constants for device changes
WM_DEVICECHANGE = 0x0219
DBT_DEVICEARRIVAL = 0x8000  # A device has been inserted (for auto detection of instruments)

class MainWindow(QMainWindow, Ui_MainWindow): # pass ui class
    def __init__(self):
        super(MainWindow, self).__init__() # inherit ui class assignments
        self.setupUi(self)

        icon = QIcon('assets/icon/icon.ico')
        self.setWindowIcon(icon)
        QApplication.setWindowIcon(icon)
        self.setWindowTitle("MALDI Control Suite")

        self.controller = FluigentController()

    def nativeEvent(self, eventType, message):
        # override to catch Windows System Messages
        msg = ctypes.wintypes.MSG.from_address(message.__int__())
        
        if msg.message == WM_DEVICECHANGE:
            if msg.wParam == DBT_DEVICEARRIVAL:
                print("USB Device Detected! Re-scanning hardware...")
                self.detect_instruments()
        
        return super().nativeEvent(eventType, message)

    def detect_instruments(self):
        self.controller.detect()
    
    def closeEvent(self, event: QCloseEvent) -> None: # gracefully exit
        try:
            # close instruments
            self.controller.close()
            pass
        except Exception as e:
            print(f'Error closing program: {e}')
        print('\nExited')

if not QApplication.instance():
    app = QApplication(sys.argv)
else:
    app = QApplication.instance()

if __name__ == '__main__':
    print(f'Running...\n')
    window = MainWindow()
    app.setStyle('Windows')
    window.show()
    app.exec()

Running...

No controllers detected!

Exited


WindowsPath('I:/Other computers/DADDY-LAPTOP/Python/UIUC/MALDI')